In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
SEED_SEQ = np.random.SeedSequence(42)
TRAIN_SEEDS = SEED_SEQ.generate_state(3)
CHECKPOINTS = [
    "004000",
    "008000",
    "012000",
    "016000",
    "020000",
    "last",
]
SCENARIOS = [
    "clean",
]
EVAL_SEED = 42
EVALUATION_DIR = "../model_evaluation"

CHECKPOINT_TO_STEP_MAPPING = {
    "000000": 0,
    "004000": 4000,
    "008000": 8000,
    "012000": 12000,
    "016000": 16000,
    "020000": 20000,
    "last": 25650,
}

In [ ]:
eval_data = {}

for scenario in SCENARIOS:
    eval_data[scenario] = {}
    for training_seed in TRAIN_SEEDS:
        eval_data[scenario][training_seed] = {}

        ckpt = "000000"
        data_path = f"{EVALUATION_DIR}/{scenario}/smolvla_base_{EVAL_SEED}"
        eval_data[scenario][training_seed][ckpt] = json.load(open(f"{data_path}/eval_info.json", "r"))

        for ckpt in CHECKPOINTS:
            data_path = f"{EVALUATION_DIR}/{scenario}/smolvla_{training_seed}_{ckpt}_{EVAL_SEED}"
            eval_data[scenario][training_seed][ckpt] = json.load(open(f"{data_path}/eval_info.json", "r"))

In [ ]:
def compute_successful_rate(max_rewards: list[float], successful_condition: float) -> float:

    return sum([r >= successful_condition for r in max_rewards]) / len(max_rewards)

successful_condition = 0.8

successful_rates = {
    scenario: {
        training_seed: {
            ckpt: compute_successful_rate(
                eval_data[scenario][training_seed][ckpt]["per_task"][0]["metrics"][
                    "max_rewards"
                ],
                successful_condition=successful_condition,
            )
            for ckpt in eval_data[scenario][training_seed]
        }
        for training_seed in eval_data[scenario]
    }
    for scenario in eval_data
}

In [ ]:
df_suc = pd.DataFrame(successful_rates["clean"])
df_suc.index = [CHECKPOINT_TO_STEP_MAPPING[ckpt] for ckpt in df_suc.index]
df_suc["max_successful_rate"] = df_suc.max(axis=1)
df_suc["mean_successful_rate"] = df_suc.mean(axis=1)
df_suc["min_successful_rate"] = df_suc.min(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(data=df_suc["mean_successful_rate"], ax=ax)
ax.fill_between(
    df_suc.index,
    df_suc["min_successful_rate"],
    df_suc["max_successful_rate"],
    alpha=0.2,
)
ax.set_title(
    "Successful Lifting Rate vs Training Steps"
    "\n"
    f"Condition for successful lifting: overlap ratio >= {successful_condition}"
)
ax.set_ylabel("Successful Lifting Rate (%)")
ax.set_xlabel("Training steps")
ax.grid()